### Functions/code order:
1. `2-ukb_target_pipe`
* Generate targets and select features by utility.
2. `Link_semmed_cuis`
* NER linking + KG linkage/graph filtering. 
* Also contains code for future DB validation/checking (what appears in future). 
* By default, output only cases with 0 kg hits. 
* NOTE! We want to keep also the feature names from this.
* Also does filtering by semantic sim, and removes features with near exact match in kg (i.e if a feature's top entity is almost identical to the feature, and that entity has a kg match, then drop feature and "children" concepts). 
* 
Outputs concepts/UMLS entities associated with features. (And 0 KG hits) .
3. `search_pubmed`
* Literature search. Also by concept/entity. 
* +- Filter by KG distance, i.e distance 2 (1 hop between target and entity). 

4. LLM - run med_Rag and rank outputs. 
* Reformats feature + disease as prompt(s), ) 
* - can run local or api model. and runs rag then sends to LLM using `MedRAG` (with modified prompt template, code)
* Note - needs extracting into terms and doing RAG, +- custom search. **Slow. **

--------------------------------
`config` = contains settings (inputs/outputs etc') for a given target/disease
* e.g. some diseases we manually added synonyms. and also define search terms (for lit search, etc')

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
# warnings.filterwarnings("W036")
import logging
logger = logging.getLogger("spacy")
logger.setLevel(logging.ERROR)
import pandas as pd

import json
import re
from tqdm import tqdm
%load_ext autoreload 
%autoreload 2 

from util import get_predictions_from_medrag, deduplicate_texts,link_kg_concepts

In [ ]:
# !pip install scispacy feature_engine biopython arfs sentence_transformers
# !pip install catboost

In [ ]:
from Link_semmed_cuis import  * #link_kg_concepts #*
from ukb_target_pipe  import  make_target_df,model_features,ipw_downsampling#*
from search_pubmed import  run_search_pubmed #*
from configs import * #config_gall

In [ ]:
# Fast_Run = True
# RUN_PIPE = False
# SAVE_OUTPUT = False

Fast_Run = False
RUN_PIPE = False#True
SAVE_OUTPUT = True
# OUTPUT_NAME= "localMini_fast.csv"

# OUTPUT_NAME="4_mini.csv"
# OUTPUT_NAME="4_full.csv"
OUTPUT_NAME = "extra.csv"

# CORPUS_DIR_CACHE_PATH = '/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/corpus'
CORPUS_DIR_CACHE_PATH = '/mnt/d/Research2/MedRAG/corpus'

In [ ]:
## import from configs
all_configs = [config_gall,
               config_celiac, config_gout,
               config_asthma,
               config_spine, config_oesophagus, config_heart, config_eye_occ,config_depression
              ] # 
# all_configs = [config_gall,config_celiac, config_gout, config_ms, config_spine, config_oesophagus, config_heart, config_eye_occ] # 
# config_ms, # no feats

# all_configs = [config_psoriasis,config_asthma,]

In [ ]:
if RUN_PIPE:
    for config in all_configs:
        try:
            print("Running pipeline for:", config["TARGET_NAME"])
            df = make_target_df(TARGET_CODES_LIST=config["TARGET_CODES_LIST"], FAST=Fast_Run,
                               REMOVE_CASES_WITH_PAST_TARGET=True
                               )
            print("-------"*5, "\nTarget extracted\n", "-------"*8)
            print(df.shape)
            # df.to_parquet(config["OUTPUT_RES_PREFIX"]+"_raw_df.parquet",index=False) # save output - not well documented
            
            if config.get("do_IPW", True):
                df = ipw_downsampling(df, K_IPW_RATIO=config.get("K_IPW_RATIO", 9))
                print(df.shape, "IPW downsampled")
            
            res_dict = model_features(df=df, FAST=Fast_Run, do_boruta_fs=config.get("do_boruta_fs", True),
                                      K_diag_thresh_value=200,
                                      SAVE_OUTPUT=True,
                                      do_mi_fs_filt=True,
                                      FEATURES_REPORT_PATH=config["FEATURES_REPORT_PATH"])

            
            link_kg_concepts(FEATURES_REPORT_PATH=config["FEATURES_REPORT_PATH"], CANDIDATE_NOVEL_CUIS_FILEPATH=config["CANDIDATE_NOVEL_CUIS_FILEPATH"],
                             TARGET_NAME=config["TARGET_NAME"],
                             SAVE_OUTPUTS=True)
            
            # Run the search and analysis pipeline
            run_search_pubmed(config)

        except Exception as e:
            print(f"Error processing {config['TARGET_NAME']}: {e}")


## Use MedRag - LLM for ranking of candidates
* 

In [ ]:
# # picks , vs all final candidates
# df_all = pd.read_csv(f"{config.get('OUTPUT_RES_PREFIX', '')}{config.get('full_results_filename', 'candidates_search_results.csv')}")
# df_all = df_all.drop_duplicates("feature_name")

In [ ]:
# ## all final candidates
# df = pd.read_csv(f"{config.get('OUTPUT_RES_PREFIX', '')}{config.get('filtered_results_filename', 'review_interesting_candidates_results.csv')}")
# print(df.shape[0],"# rows before dropping dupes by feat name")
# df = df.drop_duplicates("feature_name").dropna(axis=1,how="all")
# display(df)

In [ ]:
for config in all_configs:
    config["df"] = pd.read_csv(f"{config.get('OUTPUT_RES_PREFIX', '')}{config.get('filtered_results_filename', 'review_interesting_candidates_results.csv')}")
    print(config['OUTPUT_RES_PREFIX']+OUTPUT_NAME)
    config["df"] = config["df"].query("p_val<0.2").drop_duplicates(subset="raw_name")
    print(config["df"].shape[0])
    final_texts = deduplicate_texts(config["df"]["feature_name"], use_difflib=True, string_cutoff=0.95, distance_threshold=1)
    df = config["df"].loc[config["df"]["feature_name"].isin(final_texts)]
    print(config["df"].shape[0])
    display(config["df"].head())


In [ ]:
config["df"].nunique()

In [ ]:
import sys
# sys.path.append(r'D:\Research2\MedRAG')
# sys.path.append(r'D:\Research2\MedRAG\src')
sys.path.append('/mnt/d/Research2/MedRAG/')
sys.path.append('/mnt/d/Research2/MedRAG/src')

sys.path.append('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/')
sys.path.append('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/src')
sys.path.append('/mnt/d/Research2/MedRAG/')


## change to path with medrag and it's downloaded corpus!
os.chdir('/mnt/d/Research2/MedRAG/')
# os.chdir('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag')
from src.medrag import MedRAG

In [ ]:
def generate_medrag_prompts(data):
    results = []
    for index, row in data.iterrows():
        # Clean and prepare data
        feature_name = row['feature_name']
        raw_name = row['raw_name']
        target = row['Target']
        # Clean target string (keeping 'OR' in)
        target_clean = target.replace('(', '').replace(')', '').strip()
        # Clean feature names
        feature_name_clean = feature_name.replace('_', ' ').strip()
        raw_name_clean = raw_name.strip()
        
        # Extract relevant metadata
        p_val = row['p_val']
        correlation = row['corr']
        feature_importance = row['feature_importance']
        sim_score = row.get('sim_score', 'N/A')
        query_count = row.get('Query Count', 'N/A')
        mutual_info = row.get('MutualInfoTarget', 'N/A')
        co_occurrence_count = row.get('Co-occurrence Count', 'N/A')
        shortest_path_length = row.get('shortest_path_length', 'N/A')
        simple_path_length = row.get('simple_path_length', 'N/A')
        
        # Determine direction of effect
        if correlation > 0:
            direction = 'positive'
        elif correlation < 0:
            direction = 'negative'
        else:
            direction = 'neutral'
        
        # Prepare context (relevant documents)
        context = (
            f"Feature Name: {feature_name_clean}\n"
            f"Raw Feature Name: {raw_name_clean}\n"
            f"Target Disease(s): {target_clean}\n"
            f"Correlation: {correlation} ({direction} relationship with the target disease)\n"
            f"P-Value: {p_val}\n"
            f"Feature Importance: {feature_importance}\n"
            f"Mutual Information with Target: {mutual_info}\n"
            f"Similarity Score: {sim_score}\n"
            f"Query Count (related publications): {query_count}\n"
            f"Co-occurrence Count: {co_occurrence_count}\n"
            f"Shortest Path Length in Knowledge Graph: {shortest_path_length}\n"
            f"Simple Path Length: {simple_path_length}\n"
        )

        ## add bit about controlling for bmi, age, gender + it being a feature for prediction in advance
        # Adjusted Novelty Question and Options
        novelty_question = (
            f"Is an association (with {direction} correlation) between the feature '{feature_name_clean}' ('{raw_name_clean}') "
            f"and '{target_clean}' novel, surprising, or not well-documented in current knowledge? "
            # f"It has a correlation of {correlation} (indicating a {direction} relationship) with the target disease. "
            # f"It has with the target disease, (when controlling for age, gender, bmi). " #  (when partially controlling for age, gender, bmi)
            # f"Does it provide new insights or contradict established understanding?" #  Is it Novel?
        )
        novelty_options = {
            "A": "Yes, it is novel, provides new insights or contradicts established understanding.", # surprising, not well-documented,
            # "B": "No, it is not novel, already well-known or does not provide new insights."
            "B": "No, it is not novel, or is already well-known or established."
        }
        
        # Adjusted Plausibility Question and Options
        plausibility_question = (
            f"Does it make sense for the feature '{feature_name_clean}' (raw: '{raw_name_clean}') to be ({direction}) associated with '{target_clean}' "
            f"based on known mechanisms, pathways or theories?. " # Linear correlation is  with the target disease  (after controlling for BMI, age, gender)
            f"Is there a plausible explanation (or mechanism) for this relationship that makes sense?"
        )
        plausibility_options = {
            "A": "Yes, there is a plausible explanation for this relationship.",
            "B": "No, there is no plausible explanation for this relationship."
        }
        
        # Adjusted Utility Question and Options
        utility_question = (
            f"Assess the utility of the feature '{feature_name_clean}' (raw: '{raw_name_clean}') for predicting '{target_clean}'. "
            # f"It has a p-value of {p_val}, correlation of {correlation}, and feature importance of {feature_importance}. " # ~0 positive utility cases
            # f"It correlates {direction} with the target. " # 
            f"Does this feature potentially have practical relevance or potential utility?"
        )
        utility_options = {
            "A": "Yes, it has potential utility or practical relevance.",
            "B": "No, it lacks utility or practical relevance."
        }
        
        # Store the prompts in MedRAG format
        results.append({
            # 'feature': feature_name_clean,
            # 'raw_feature': raw_name_clean,
           'feature':  row['feature_name'],
            'raw_feature': row['raw_name'],
            "target": row['Target'],
            'context': context,
            'novelty_question': novelty_question,
            'novelty_options': novelty_options,
            'plausibility_question': plausibility_question,
            'plausibility_options': plausibility_options,
            'utility_question': utility_question,
            'utility_options': utility_options
        })
        
    return results


In [ ]:
from src.medrag import MedRAG
import pandas as pd
from tqdm import tqdm
import re
from sklearn.metrics import classification_report, roc_auc_score
import gc
import json

In [ ]:
LL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct" # works, less stable than 3.0 model?
# LL_NAME ="meta-llama/Llama-3.2-3B-Instruct"
# LL_NAME ="meta-llama/Llama-3.2-1B-Instruct"
# LL_NAME ="mistralai/Ministral-8B-Instruct-2410"
 # LL_NAME =  = "meta-llama/Meta-Llama-3-70B-Instruct" 
# LL_NAME = "aaditya/Llama3-OpenBioLLM-8B" # unsupported template
# LL_NAME = "OpenScholar/Llama-3.1_OpenScholar-8B"  # openscholar moidel # error loading? 

# LL_NAME ="allenai/Llama-3.1-Tulu-3-8B"

# LL_NAME ="OpenAI/gpt-4o-mini"
# LL_NAME ="OpenAI/gpt-4o" # hits token limits on tier 1 account. tokens per min (TPM): Limit 30000, Used 24125, Requested 7561. https://platform.openai.com/account/rate-limits

# if Fast_Run:
    # LL_NAME ="meta-llama/Llama-3.2-1B-Instruct"
    # LL_NAME ="meta-llama/Llama-3.2-3B-Instruct"

LL_NAME

### plausible

In [ ]:
# %%time
# plausible_pred = []
# plausible_cot = []
# plausible_answers_full = []
# plausible_answers_json = []
# answer_snippets = []
# options = results[0]["plausibility_options"]

# # for index, row in tqdm(df2.iterrows(), total=df2.shape[0], desc="Processing rows"): # when using df
# for index, row in enumerate(tqdm(results, total=len(results), desc="Processing rows")):
#     question = row['plausibility_question']
#     answer, snippets, scores = medrag.answer(question=question, options=options, k=2 if Fast_Run else 28) #32
#     # Ensure the answer ends with a single closing curly brace
#     if not answer.endswith("}"): ## what about " at end?
#         if "}" not in answer:
#             answer += "}" 
    
#     # Use regex to replace two or more consecutive closing curly braces with just one
#     answer = re.sub(r'\}+', '}', answer)
    
#     try:
#         # json_ans = json.loads(re.search('{.+}', answer, re.IGNORECASE).group(0))
#         json_ans = json.loads(re.search(r'{.*?}', answer,  re.DOTALL | re.IGNORECASE).group(0))
        
#     except:
#         json_ans = json.dumps({'step_by_step_thinking':None,'answer_choice':None})
#         print("Failed parse")
#         print(answer)

#     try:
#         cot = json_ans['step_by_step_thinking']
#         pred = json_ans['answer_choice']
#         print(cot)
#     except:
#         pred =answer.split('answer_choice')[-1] 
#         cot = answer.split("step_by_step_thinking")[-1].split("answer_choice")[0]

#     plausible_pred.append(pred)
#     plausible_cot.append(cot)
#     plausible_answers_full.append(answer)
#     # plausible_answers_json.append(json_ans)
#     answer_snippets.append(snippets)

# #### alt json parsing: may cut too much!!

# # # Initialize lists
# # plausible_pred = []
# # plausible_cot = []
# # plausible_answers_full = []
# # answer_snippets = []
# # options = results[0]["plausibility_options"]

# # for index, row in enumerate(tqdm(results, total=len(results), desc="Processing rows")):
# #     question = row['plausibility_question']
# #     answer, snippets, scores = medrag.answer(question=question, options=options, k=2 if Fast_Run else 30)

# #     # Clean the LLM output
# #     answer = answer.strip()

# #         ## add:
# #     #     # Ensure the answer ends with a single closing curly brace
# #     if not answer.endswith("}"): ## what about " at end?
# #         answer += "}" 
                
# #     # Remove code block markers (like ```json, ```)
# #     # This will remove the markers without affecting the content
# #     answer = re.sub(r'^```(?:json)?\s*', '', answer, flags=re.MULTILINE)
# #     answer = re.sub(r'^```\s*', '', answer, flags=re.MULTILINE)
# #     answer = re.sub(r'\s*```$', '', answer)
# #     answer = answer.strip()

# #     # Extract the JSON part from the text
# #     json_match = re.search(r'\{.*\}', answer, re.DOTALL)
# #     if json_match:
# #         json_str = json_match.group(0)
# #         try:
# #             json_ans = json.loads(json_str)
# #         except json.JSONDecodeError as e:
# #             print(f"JSON parsing error: {e}")
# #             print("Attempting to fix common issues...")
# #             # Attempt to fix common JSON issues
# #             json_str = json_str.replace('\n', ' ')
# #             json_str = re.sub(r',\s*}', '}', json_str)
# #             json_str = re.sub(r',\s*\]', ']', json_str)

# #             try:
# #                 json_ans = json.loads(json_str)
# #             except json.JSONDecodeError as e:
# #                 json_ans = {'step_by_step_thinking': None, 'answer_choice': None}
# #                 print("Failed to parse JSON after correction:")
# #                 print(json_str)
# #         except Exception as e:
# #             json_ans = {'step_by_step_thinking': None, 'answer_choice': None}
# #             print("Unexpected error during JSON parsing:", e)
# #     else:
# #         # No JSON object found
# #         json_ans = {'step_by_step_thinking': None, 'answer_choice': None}
# #         print("No JSON object found in the answer:")
# #         print(answer)

# #     # Extract 'step_by_step_thinking' and 'answer_choice'
# #     cot = json_ans.get('step_by_step_thinking')
# #     pred = json_ans.get('answer_choice')

# #     # Append to lists
# #     plausible_pred.append(pred)
# #     plausible_cot.append(cot)
# #     plausible_answers_full.append(answer)
# #     answer_snippets.append(snippets)


In [ ]:
# ### Temp save snippets to save on retrieval / copy/backup in case of failure
# if SAVE_OUTPUT:
#     pd.DataFrame({"answer_snippets":answer_snippets}).to_csv("temp_answer_snippets.csv")

In [ ]:
# # Function to get predictions from MedRAG # get_predictions
def get_predictions_from_medrag(medrag, results, question_key, options_key, snippets=None,return_snippets=False):
    predictions = []
    explanations = []
    if return_snippets: snippets_list = []
    for index, row in enumerate(tqdm((results), total=len(results), desc="Processing rows")):
    # for index, row in enumerate(results):
        question = row[question_key]
        print(question)
        options = row[options_key]
        snippet = snippets[index] if snippets is not None else None
        answer, retrieved_snippets, _ = medrag.answer(question=question, options=options, snippets=snippet,
                                                      k=4 if FAST_RUN else 32)
        print(answer)
        try:
            if not answer.endswith("}"): ## what about " at end?
                if "}" not in answer:
                    answer += "}" 
            # Use regex to replace two or more consecutive closing curly braces with just one
            answer = re.sub(r'\}+', '}', answer)

            # Fix missing commas between key-value pairs
            answer = re.sub(r'(")\s*(\n\s*)?(?="[\w_]+":)', r'\1,\2,', answer)
            # # replace all occurrences of single quote with double quote in the JSON string s and in the latter case will not replace escaped single-quotes.
            # p = re.compile('(?<!\\\\)\'')
            # answer = p.sub('\"', answer)
            json_ans = json.loads(re.search(r'{.*?}', answer,  re.DOTALL | re.IGNORECASE).group(0))
            pred = json_ans.get('answer_choice', None)
            predictions.append(pred)
            explanations.append(json_ans.get('step_by_step_thinking', None))
        except Exception as e:
            # replace all occurrences of single quote with double quote in the JSON string s and in the latter case will not replace escaped single-quotes.
            p = re.compile('(?<!\\\\)\'')
            answer = p.sub('\"', answer)
            try:
                json_ans = json.loads(re.search(r'{.*?}', answer,  re.DOTALL | re.IGNORECASE).group(0))
                pred = json_ans.get('answer_choice', None)
                predictions.append(pred)
                explanations.append(json_ans.get('step_by_step_thinking', None))
            except:
                predictions.append(None)
                explanations.append( None)
            # predictions.append(None)
                print(f"Failed to parse answer: {e}\n{answer}")
        if return_snippets: snippets_list.append(retrieved_snippets)
    if return_snippets:
        return predictions,explanations,snippets_list
    else:
        return predictions,explanations

# Main function to run MedRAG and save results
def run_medrag_pipeline(config,LL_NAME:str="meta-llama/Llama-3.2-3B-Instruct",FastMode=False,save=False,snippets = None):
    # df = pd.read_csv(f"{config.get('OUTPUT_RES_PREFIX', '')}{config.get('filtered_results_filename', 'review_interesting_candidates_results.csv')}")
    df = config["df"]

    df = df.drop_duplicates("feature_name").dropna(axis=1, how="all")
    if FastMode: df = df.sample(5).copy()
    
    results = generate_medrag_prompts(df)

    if FastMode:
        results = results[-2:]
        medrag = MedRAG(llm_name=LL_NAME, rag=True, retriever_name="BM25",
                    HNSW=True , 
                        # corpus_name="MedText",
                    db_dir= CORPUS_DIR_CACHE_PATH,
                    corpus_name="Textbooks"
                    )
    else:
        # medrag = MedRAG(llm_name=LL_NAME, rag=True, retriever_name="BM25",
        #             corpus_name= "MedText"
        #                 # corpus_name="MedCorp" 
        #                )

        medrag = MedRAG(llm_name=LL_NAME, rag=True,
                retriever_name="BM25", # "BM25" #"RRF-2"#"MedCPT" - RRF2 woudl take ~ 11 hours for pubmed
                # retriever_name="RRF-2" ## fast enough with small data
                 # corpus_name="MedText"  ## small 
                 corpus_name="PubMed" 
                ,HNSW=True 
                # ,corpus_cache=True
                ,db_dir= CORPUS_DIR_CACHE_PATH#'/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/corpus',
               )
    
    # Load cached snippets if available
    # snippets = None
    # if os.path.exists(config['cached_snippets_file']):
    ## alt: config['OUTPUT_RES_PREFIX']+"snippets.csv"
    #     snippets_df = pd.read_csv(config['cached_snippets_file'])
    #     snippets = snippets_df['answer_snippets'].tolist()
    
    # novel_pred,explanations_novel,snippets = get_predictions_from_medrag(medrag, results, 'novelty_question', 'novelty_options',
    #                                                                      # snippets=snippets,
    #                                                                      return_snippets=True)
    novel_pred,explanations_novel = get_predictions_from_medrag(medrag, results, 'novelty_question', 'novelty_options',
                                                                         # snippets=snippets,
                                                                         return_snippets=False)
    plausible_pred,explanations_plaus = get_predictions_from_medrag(medrag, results, 'plausibility_question', 'plausibility_options',
                                                                    snippets=snippets
                                                                   )
    utility_pred,explanations_util = get_predictions_from_medrag(medrag, results, 'utility_question', 'utility_options',
                                                                 snippets=snippets
                                                                )
    
    df_results_all = pd.DataFrame({
        'feature': [x['feature'] for x in results],
        "target": [x['target'] for x in results],
        "novel_pred": novel_pred,
        "plausible_pred": plausible_pred,
        "utility_pred": utility_pred,
        "novel_cot": explanations_novel,
        "plausible_cot": explanations_plaus,
        "utility_cot": explanations_util,
        # "snippets": snippets if snippets else [None] * len(results)
    })
    try:
        df_results_all["novel"] = df_results_all["novel_pred"].astype(str).str.contains("A").astype(int)
        df_results_all["plausible"] = df_results_all["plausible_pred"].astype(str).str.contains("A").astype(int)
        df_results_all["utility"] = df_results_all["utility_pred"].astype(str).str.contains("A").astype(int)
    except:()
    df_results_all["snippets"] = snippets if snippets else [None] * len(results) # put at end, messes up extraction # change: unindented
    if save:
        df_results_all.to_csv(config['OUTPUT_RES_PREFIX']+OUTPUT_NAME, index=False)
        if snippets is not None:
            # pd.DataFrame({"answer_snippets": snippets}).to_csv(config['cached_snippets_file'], index=False)
            pd.DataFrame({"answer_snippets": snippets}).to_csv(config['OUTPUT_RES_PREFIX']+"snippets.csv")
    display(df_results_all)
    return df_results_all


# df_results_all = run_medrag_pipeline(config,LL_NAME="meta-llama/Llama-3.2-1B-Instruct",FastMode=True)
# df_results_all = run_medrag_pipeline(config,LL_NAME=LL_NAME,FastMode=True)

# # # Running the pipeline
# if __name__ == "__main__":
#     # run_pipeline(CONFIG)
import sys
# sys.path.append(r'D:\Research2\MedRAG')
# sys.path.append(r'D:\Research2\MedRAG\src')
sys.path.append('/mnt/d/Research2/MedRAG/')
sys.path.append('/mnt/d/Research2/MedRAG/src')

sys.path.append('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/')
sys.path.append('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/src')
# sys.path.append('/mnt/d/Research2/MedRAG/corpus/')
## change to path with medrag and it's downloaded corpus!
# os.chdir('/mnt/d/Research2/MedRAG/')
# os.chdir('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag')
from src.medrag import MedRAG
from tqdm import tqdm
import json
from util import get_predictions_from_medrag

# CORPUS_DIR_CACHE_PATH = '/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/corpus'
# CORPUS_DIR_CACHE_PATH = '/mnt/d/Research2/MedRAG/corpus'
# ## change to path with medrag and it's downloaded corpus!
# # os.chdir('/mnt/d/Research2/MedRAG/')
# os.chdir('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag')
# run_medrag_pipeline(config)

In [ ]:
# SAVE_OUTPUT=False

In [ ]:
plus_configs = [
    # config_gall,
               # config_celiac, config_gout, config_oesophagus,
               # config_spine, config_oesophagus, config_heart, config_eye_occ,config_depression,
    config_psoriasis,
     # config_asthma,
              ]
for config in plus_configs:
    # snippets = pd.read_csv(config['OUTPUT_RES_PREFIX']+"4_mini.csv")["snippets"] #OUTPUT_NAME)
    # snippets = pd.read_csv(config['OUTPUT_RES_PREFIX']+"8b.csv")["snippets"]
    print(config['OUTPUT_RES_PREFIX'])
    config["df_lm"] = run_medrag_pipeline(config,LL_NAME=LL_NAME,
                                          # FastMode=True
                                          FastMode=Fast_Run,
                                          save=SAVE_OUTPUT
                                         ) 
    print(config["df_lm"])
    if SAVE_OUTPUT:
        config["df_lm"][['feature',  'novel', 'plausible', 'utility','novel_cot',
                         'plausible_cot', 'utility_cot','target', ]].to_csv(config['OUTPUT_RES_PREFIX']+"8b_llm_explain.csv",index=False)

In [ ]:
# print(config["df_lm"].loc[config["df_lm"][[ 'novel', 'plausible']].max(axis=1)>0].shape)
# config["df_lm"].loc[config["df_lm"][[ 'novel', 'plausible']].max(axis=1)>0][['feature',  'novel', 'plausible', 'utility','novel_cot',
#                          'plausible_cot', 'utility_cot','target', ]].to_csv(config['OUTPUT_RES_PREFIX']+"8b_llm_explain_candidates.csv",index=False)

In [ ]:
config["df_lm"]

In [ ]:
"""
for config in all_configs:
    config["df_lm"][['feature', 'target',  'novel',
       'plausible', 'utility', 'novel_cot', 'plausible_cot', 'utility_cot', ]].sort_values(by=[ 'novel',
       'plausible', 'utility'],ascending=False).to_csv(config['OUTPUT_RES_PREFIX']+"_gpt4_v2.csv",index=False)
"""

In [ ]:
config["df_lm"].columns